# VisoMaster Setup and Path Fixes

This notebook will:
1. Navigate to the correct VisoMaster folder
2. Fix backslash paths in Models.py
3. Install dependencies from existing requirements file
4. Run the existing download_models.py script

Current Date: 2025-05-01 14:21:48 UTC  
User: remphan1618

## 1. Finding and Navigating to the VisoMaster Folder

In [ ]:
# Check workspace for VisoMaster (case-sensitive check)
!ls -la /workspace/

In [ ]:
# Create symbolic link to handle case sensitivity
!if [ -d "/workspace/VisoMaster" ] && [ ! -d "/workspace/visomaster" ]; then \
    ln -sf /workspace/VisoMaster /workspace/visomaster; \
    echo "Created symbolic link from /workspace/VisoMaster to /workspace/visomaster"; \
fi

In [ ]:
# Navigate to VisoMaster folder
%cd /workspace/visomaster
!pwd

In [ ]:
# List repository contents to verify
!ls -la

## 2. Fixing Backslash Paths in Models.py

Windows-style paths with backslashes need to be converted to forward slashes for Linux compatibility.

In [ ]:
# First, let's check if Models.py exists and where it is
!find /workspace/visomaster -name "Models.py"

In [ ]:
# Backup the file before modifying it
!find /workspace/visomaster -name "Models.py" -exec cp {} {}.bak \;

In [ ]:
# Create a Python function to fix the file
def fix_backslashes_in_file(file_path):
    try:
        # Read the file content
        with open(file_path, 'r') as file:
            content = file.read()
        
        # Replace backslashes with forward slashes
        # Handles string patterns like r"path\to\file" or "path\to\file"
        import re
        # Find all strings with backslashes
        patterns = [
            r'r"([^"]*\\[^"]*)"',  # Raw strings: r"path\to\file"
            r'"([^"]*\\[^"]*)"',   # Regular strings: "path\to\file"
            r"r'([^']*\\[^']*)'\s",   # Raw strings with single quotes: r'path\to\file'
            r"'([^']*\\[^']*)'\s"    # Regular strings with single quotes: 'path\to\file'
        ]
        
        # Process each pattern
        for pattern in patterns:
            matches = re.findall(pattern, content)
            for match in matches:
                fixed_path = match.replace('\\', '/')
                if 'r"' + match + '"' in content:
                    content = content.replace('r"' + match + '"', '"' + fixed_path + '"')
                elif '"' + match + '"' in content:
                    content = content.replace('"' + match + '"', '"' + fixed_path + '"')
                elif "r'" + match + "'" in content:
                    content = content.replace("r'" + match + "'", "'" + fixed_path + "'")
                elif "'" + match + "'" in content:
                    content = content.replace("'" + match + "'", "'" + fixed_path + "'")
        
        # Save the modified content back to the file
        with open(file_path, 'w') as file:
            file.write(content)
            
        print(f"Successfully fixed backslashes in {file_path}")
        return True
    except Exception as e:
        print(f"Error fixing backslashes in {file_path}: {e}")
        return False

In [ ]:
# Get the path(s) to Models.py
import subprocess
import os

result = subprocess.run(['find', '/workspace/visomaster', '-name', 'Models.py'], 
                        stdout=subprocess.PIPE, text=True)
model_paths = result.stdout.strip().split('\n')

for path in model_paths:
    if path:  # Skip empty paths
        print(f"Fixing file: {path}")
        if os.path.exists(path):
            fix_backslashes_in_file(path)
        else:
            print(f"File not found: {path}")

if not model_paths or not model_paths[0]:
    print("No Models.py file found")

## 3. Installing Dependencies

Installing scikit-image and other dependencies from the existing requirements file.

In [ ]:
# Check if requirements_cu124.txt exists and show its contents
!find /workspace/visomaster -name "requirements_cu124.txt" -exec cat {} \;

In [ ]:
# Activate the visomaster conda environment
import os
os.environ['PATH'] = '/opt/conda/envs/visomaster/bin:' + os.environ['PATH']

# Install scikit-image with conda
!conda install -y scikit-image

In [ ]:
# Install required packages from requirements_cu124.txt
req_file = subprocess.run(['find', '/workspace/visomaster', '-name', 'requirements_cu124.txt'], 
                          stdout=subprocess.PIPE, text=True).stdout.strip().split('\n')[0]

if req_file:
    print(f"Installing requirements from {req_file}")
    !pip install -r $req_file
else:
    print("requirements_cu124.txt not found")

## 4. Running Model Download Script

Finding and executing the existing download_models.py script.

In [ ]:
# Find download_models.py script
!find /workspace/visomaster -name "download_models.py"

In [ ]:
# Run the download_models.py script
model_script = subprocess.run(['find', '/workspace/visomaster', '-name', 'download_models.py'], 
                              stdout=subprocess.PIPE, text=True).stdout.strip().split('\n')[0]

if model_script:
    print(f"Running model download script: {model_script}")
    # Change to the directory containing the script
    script_dir = os.path.dirname(model_script)
    %cd $script_dir
    !python $model_script
else:
    print("download_models.py not found")

## 5. Test Running the main.py Script

In [ ]:
# Find the main.py script
!find /workspace/visomaster -name "main.py"

In [ ]:
# Test if main.py can be imported without errors
main_py = subprocess.run(['find', '/workspace/visomaster', '-name', 'main.py'], 
                          stdout=subprocess.PIPE, text=True).stdout.strip().split('\n')[0]

if main_py:
    main_dir = os.path.dirname(main_py)
    %cd $main_dir
    try:
        import importlib.util
        spec = importlib.util.spec_from_file_location("main", main_py)
        main_module = importlib.util.module_from_spec(spec)
        print(f"Successfully imported main.py from {main_py}")
    except Exception as e:
        print(f"Error importing main.py: {e}")
else:
    print("main.py not found")

## 6. Verify Repository Structure

In [ ]:
# Get a file listing of the repository to verify key files
!find /workspace/visomaster -type f -name "*.py" | sort

## 7. Summary of Fixes

This notebook has:

1. Fixed case sensitivity issues by creating a symbolic link between VisoMaster and visomaster
2. Fixed backslash paths in Models.py for Linux compatibility
3. Installed scikit-image with conda
4. Installed dependencies from the existing requirements_cu124.txt file
5. Run the existing download_models.py script to download model assets

Your VisoMaster folder is now properly set up with all backslash paths fixed and dependencies installed. The application should now run correctly with proper path handling.